# Name That LLM: five baselines and the first offline A/B test

**Goal:** score five classic text classifiers *the same honest way*, then decide with
statistics, not gut feeling, which one becomes our baseline **champion**.

Part of a learning-in-public, end-to-end MLOps project:
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm). All the
modelling code comes from the project's package (pinned to a git tag), so this notebook is
short. It only orchestrates and explains.

**How every model is scored (the backtest):** 5-fold cross-validation **grouped by prompt**.
Each model is always scored on prompts it never saw, which is exactly the situation on the
hidden test. All five models use the *same* folds, so their scores are directly comparable.

## 1. Setup

We install the package from tag `v0.2.0`, and connect Weights & Biases (W&B) if the
`WANDB_API_KEY` secret is attached to this notebook. Without the secret, everything still
runs; only the tracking is skipped. Tracking must never be a single point of failure.

In [ ]:
# A tag never moves, so this notebook always runs the exact code it was written for.
%pip install -q "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@v0.2.0"

In [ ]:
import os
import time

import numpy as np
import pandas as pd

from name_that_llm.config import load_config
from name_that_llm.data import load_train
from name_that_llm.metrics import lab_accuracy, prompt_bootstrap_ci
from name_that_llm.train import run_cv

# Kaggle Secrets keep the key out of the notebook's code (which is public).
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    TRACK = True
except Exception as exc:  # no secret attached, or not running on Kaggle
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}). Results are still computed.")

if TRACK:
    from name_that_llm.tracking import log_cv_run

train = load_train()
print("train:", train.shape, "| tracking:", TRACK)

## 2. Run the five baselines

| config | model | idea |
|---|---|---|
| `mnb` | Multinomial Naive Bayes | how often each model uses each n-gram |
| `cnb` | Complement Naive Bayes | learns from all the *other* classes; often steadier on text |
| `lr` | Logistic regression | the standard strong linear text classifier |
| `svm` | Linear SVM + calibration | the community's reference baseline, made to output probabilities |
| `nbsvm` | NB-SVM | Naive Bayes feature weights fed into a linear model |

Every model sees the same features: word 1–2-grams plus character 1–5-grams (TF-IDF).

In [ ]:
CONFIGS = ["mnb", "cnb", "lr", "svm", "nbsvm"]
y = train["model"].to_numpy()
groups = train["prompt_id"].to_numpy()

results, rows = {}, []
for name in CONFIGS:
    cfg = load_config(name)
    start = time.time()
    result = run_cv(cfg, train)
    seconds = time.time() - start
    results[name] = result
    rows.append(
        {
            "config": name,
            "cv_balanced_accuracy": result.score,
            "fold_std": float(np.std(result.fold_scores)),
            "lab_accuracy": lab_accuracy(y, result.pred),
            "minutes": seconds / 60,
            "wandb_run": log_cv_run(cfg, result, train) if TRACK else "-",
        }
    )
    print(f"{name:6s} CV {result.score:.4f}  ({seconds / 60:.1f} min)")

table = pd.DataFrame(rows).sort_values("cv_balanced_accuracy", ascending=False)
table.round(4)

In [ ]:
import matplotlib.pyplot as plt

# One dot per fold, plus a bar for the mean. The spread between folds shows how much a
# score would move if we had drawn different prompts, i.e. how much to trust a small gap.
INK, MUTED, SERIES_1, SURFACE = "#0b0b0b", "#898781", "#2a78d6", "#fcfcfb"
order = table["config"].tolist()[::-1]  # best on top
fig, ax = plt.subplots(figsize=(8, 3.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for i, name in enumerate(order):
    folds = results[name].fold_scores
    ax.scatter(folds, [i] * len(folds), s=28, color=SERIES_1, alpha=0.55, zorder=3)
    ax.plot([results[name].score] * 2, [i - 0.3, i + 0.3], color=INK, linewidth=2.5, zorder=4)
ax.set_yticks(range(len(order)), order)
ax.set_xlabel("balanced accuracy (dots: the 5 folds, bar: overall)")
ax.set_title("Five baselines on identical prompt-grouped folds", loc="left", fontweight="bold")
ax.grid(axis="x", color="#e1e0d9", linewidth=0.6)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()

## 3. The first offline A/B test: is the winner *really* better?

The two best models are a few points apart, but the folds also move by a few points. So is
the gap real, or luck with *these* 150 prompts?

The **paired prompt bootstrap** answers that. It re-draws the 150 prompts with replacement
2,000 times and recomputes "winner minus runner-up" on each re-draw. Both models are scored
on the same re-drawn prompts (that's the "paired" part). The middle 95% of those
differences is the **confidence interval**:
- **The whole interval is above 0:** the winner is better with about 95% confidence. Promote it.
- **The interval contains 0:** we can't tell them apart. Prefer the simpler or faster model.

In [ ]:
best, runner_up = table["config"].iloc[0], table["config"].iloc[1]
diff, lo, hi = prompt_bootstrap_ci(
    y, results[runner_up].pred, results[best].pred, groups, n_boot=2000, seed=0
)
print(f"{best} minus {runner_up}: {diff:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
print(
    "verdict:",
    f"{best} is better" if lo > 0 else "can't tell them apart: prefer the simpler/faster",
)

## 4. What does the classifier look at?

We fit logistic regression on all the training data and list, for one model per lab, the
features with the largest positive weights, i.e. what pushes the prediction *towards* that
model. Character features are shown between `[` `]` so spaces and punctuation stay visible.

In [ ]:
from name_that_llm.models import build_model

lr = build_model(load_config("lr")).fit(train["text"], train["model"])
names = lr.named_steps["features"].get_feature_names_out()
coefs = lr.named_steps["clf"].coef_
classes = list(lr.named_steps["clf"].classes_)


def pretty(feature):
    kind, _, value = feature.partition("__")
    return f"[{value}]" if kind == "chars" else value


SHOWN = [
    "claude-opus-5",
    "gemini-3.1-pro-preview",
    "kimi-k3",
    "gpt-5.6-luna",
    "qwen3.8-max",
    "glm-5.3",
]
top = {m: [pretty(names[j]) for j in np.argsort(coefs[classes.index(m)])[::-1][:10]] for m in SHOWN}
pd.DataFrame(top, index=pd.RangeIndex(1, 11, name="rank"))

## 5. Findings

*Filled in after the run.*